In [ ]:
"""Imports and plotting setup"""

import os
import time
import json
import csv
import math
import platform
import datetime
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats.qmc import LatinHypercube

try:
    import psutil
except ImportError:
    psutil = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(1234)
torch.manual_seed(1234)

OUTPUT_DIR = r"./open_source_assets"
os.makedirs(OUTPUT_DIR, exist_ok=True)
COST_REPORT_PREFIX = "lid_driven_cavity_porous_bed_single_network_autopinns"

"""Physical parameters and data preparation"""
STAGE1_EPOCHS = 10000
STAGE2_EPOCHS = 10000

N_BASE_DARCY = 5000
N_BASE_STOKES = 5000
N_BC_DARCY = 1200
N_BC_STOKES = 1600
N_INTERFACE = 1200
N_T0_DARCY = 1200
N_T0_STOKES = 1200

X_MIN, X_MAX = 0.0, 1.0
Y_D_MIN, Y_D_MAX = -0.5, 0.0
Y_NS_MIN, Y_NS_MAX = 0.0, 0.5
T_MIN, T_MAX = 0.0, 1.0

K = [[1.0e-5, 0.0], [0.0, 1.0e-5]]
nu = 1.0e-2
g = 1.0
alpha = 1.0
s = 1.0
trace = K[0][0] + K[1][1]

def sync_cuda_if_available():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        sync_cuda_if_available()
        torch.cuda.reset_peak_memory_stats()

def get_cuda_memory_summary():
    if not torch.cuda.is_available():
        return {
            "cuda_available": False,
            "current_allocated_mb": None,
            "current_reserved_mb": None,
            "peak_allocated_mb": None,
            "peak_reserved_mb": None,
        }
    sync_cuda_if_available()
    return {
        "cuda_available": True,
        "current_allocated_mb": torch.cuda.memory_allocated() / 1024**2,
        "current_reserved_mb": torch.cuda.memory_reserved() / 1024**2,
        "peak_allocated_mb": torch.cuda.max_memory_allocated() / 1024**2,
        "peak_reserved_mb": torch.cuda.max_memory_reserved() / 1024**2,
    }

def get_cpu_memory_mb():
    if psutil is None:
        return None
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def get_hardware_info():
    info = {
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python_version": platform.python_version(),
        "pytorch_version": torch.__version__,
        "device": str(device),
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cpu_count_logical": os.cpu_count(),
        "cpu_rss_mb_at_report": get_cpu_memory_mb(),
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        info.update({
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_count": torch.cuda.device_count(),
            "gpu_total_memory_mb": props.total_memory / 1024**2,
            "gpu_compute_capability": f"{props.major}.{props.minor}",
        })
    else:
        info.update({
            "gpu_name": None,
            "gpu_count": 0,
            "gpu_total_memory_mb": None,
            "gpu_compute_capability": None,
        })
    return info

def count_trainable_parameters(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

def _json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)

def write_computational_cost_report(report, prefix=COST_REPORT_PREFIX):
    json_path = os.path.join(OUTPUT_DIR, f"{prefix}_computational_cost.json")
    csv_path = os.path.join(OUTPUT_DIR, f"{prefix}_computational_cost_summary.csv")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2, default=_json_default)

    training = report.get("training", {})
    inference = report.get("inference", {})
    rl = report.get("rl_overhead", {})
    model_info = report.get("model", {})
    memory = report.get("memory", {})
    hardware = report.get("hardware", {})
    summary = {
        "method": report.get("method"),
        "gpu_name": hardware.get("gpu_name"),
        "trainable_parameters": model_info.get("trainable_parameters"),
        "stage1_time_s": training.get("stage1_time_s"),
        "stage2_time_s": training.get("stage2_time_s"),
        "training_time_s": training.get("total_training_time_s"),
        "training_gpu_hours": training.get("gpu_hours"),
        "peak_gpu_memory_allocated_mb": memory.get("training_peak_allocated_mb"),
        "peak_gpu_memory_reserved_mb": memory.get("training_peak_reserved_mb"),
        "rl_inference_time_s": rl.get("policy_inference_time_s"),
        "rl_update_time_s": rl.get("policy_update_time_s"),
        "rl_total_overhead_time_s": rl.get("total_measured_overhead_time_s"),
        "rl_overhead_fraction": rl.get("overhead_fraction_of_training_time"),
        "prediction_time_s": inference.get("prediction_time_s"),
        "prediction_points": inference.get("prediction_points"),
    }
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        f.write(",".join(summary.keys()) + "\n")
        f.write(",".join("" if v is None else str(v) for v in summary.values()) + "\n")
    print(f"Cost report saved:\n  {json_path}\n  {csv_path}")
    return {"json": json_path, "csv": csv_path}

def lhs_box(n, x0, x1, y0, y1, t0=T_MIN, t1=T_MAX):
    sampler = LatinHypercube(d=3)
    z = sampler.random(n)
    return z * np.array([x1 - x0, y1 - y0, t1 - t0]) + np.array([x0, y0, t0])

def sample_line(n, side):
    r = np.random.rand(n, 1)
    tt = np.random.rand(n, 1)
    if side == "top":
        x, y = r, np.ones_like(r) * Y_NS_MAX
    elif side == "left_s":
        x, y = np.zeros_like(r), r * (Y_NS_MAX - Y_NS_MIN) + Y_NS_MIN
    elif side == "right_s":
        x, y = np.ones_like(r), r * (Y_NS_MAX - Y_NS_MIN) + Y_NS_MIN
    elif side == "left_d":
        x, y = np.zeros_like(r), r * (Y_D_MAX - Y_D_MIN) + Y_D_MIN
    elif side == "right_d":
        x, y = np.ones_like(r), r * (Y_D_MAX - Y_D_MIN) + Y_D_MIN
    elif side == "bottom":
        x, y = r, np.ones_like(r) * Y_D_MIN
    elif side == "interface":
        x, y = r, np.zeros_like(r)
    else:
        raise ValueError(side)
    return np.hstack([x, y, tt])

def to_tensor_xyzt(X, requires_grad=True):
    x = torch.tensor(X[:, 0:1], dtype=torch.float32, requires_grad=requires_grad).to(device)
    y = torch.tensor(X[:, 1:2], dtype=torch.float32, requires_grad=requires_grad).to(device)
    t = torch.tensor(X[:, 2:3], dtype=torch.float32, requires_grad=requires_grad).to(device)
    return x, y, t

"""Reward functions"""
def calculate_reward(current_losses, previous_losses, initial_losses):
    if previous_losses is None:
        return 0.0

    total_current = sum(current_losses)
    total_previous = sum(previous_losses)

    if total_previous == 0:
        return 0.0

    improvement = (total_previous - total_current) / total_previous

    if improvement > 0.08:
        base_reward = 2.5 * improvement
    elif improvement > 0.04:
        base_reward = 2.2 * improvement
    elif improvement > 0.02:
        base_reward = 2.0 * improvement
    elif improvement > 0.01:
        base_reward = 1.8 * improvement
    elif improvement > -0.01:
        base_reward = 0.0
    else:
        base_reward = -2.0 * abs(improvement)

    if total_current < 0.05:
        absolute_reward = 1.0
    elif total_current < 0.1:
        absolute_reward = 0.8
    elif total_current < 0.5:
        absolute_reward = 0.5
    elif total_current < 1.0:
        absolute_reward = 0.2
    elif total_current > 20.0:
        absolute_reward = -0.3
    elif total_current > 50.0:
        absolute_reward = -0.5
    else:
        absolute_reward = 0.0

    balance_reward = 0.0
    if len(current_losses) > 1:
        max_loss = max(current_losses)
        min_loss = min(current_losses)
        if max_loss > 0:
            balance_ratio = min_loss / max_loss
            if balance_ratio > 0.1:
                balance_reward = 0.3
            elif balance_ratio < 0.01:
                balance_reward = -0.3

    stability_reward = 0.0
    if abs(improvement) < 0.005:
        if total_current < 1.0:
            stability_reward = 0.2

    total_reward = base_reward + absolute_reward + balance_reward + stability_reward
    return max(min(total_reward, 3.0), -3.0)

def calculate_arch_reward(pre_losses, post_losses, total_params, action_type=None,
                         current_depth=None, current_units=None):
    pre_total = sum(pre_losses)
    post_total = sum(post_losses)

    if pre_total == 0:
        return 0.0

    improvement = (pre_total - post_total) / pre_total

    if improvement > 0.10:
        base_reward = 2.5 * improvement
    elif improvement > 0.06:
        base_reward = 2.2 * improvement
    elif improvement > 0.03:
        base_reward = 2.0 * improvement
    elif improvement > 0.01:
        base_reward = 1.8 * improvement
    elif improvement > -0.02:
        base_reward = 0.0
    else:
        base_reward = -2.2 * abs(improvement)

    if post_total < 0.05:
        absolute_reward = 1.0
    elif post_total < 0.1:
        absolute_reward = 0.8
    elif post_total < 0.5:
        absolute_reward = 0.5
    elif post_total < 1.0:
        absolute_reward = 0.2
    elif post_total > 20.0:
        absolute_reward = -0.3
    elif post_total > 50.0:
        absolute_reward = -0.5
    else:
        absolute_reward = 0.0

    param_reward = 0.0
    if total_params > 500000:
        param_reward = -0.5
    elif total_params > 300000:
        param_reward = -0.3
    elif total_params < 100000:
        param_reward = 0.2

    total_reward = base_reward + absolute_reward + param_reward
    return max(min(total_reward, 3.0), -3.0)

"""RL controller"""
class RLController:
    def __init__(self, n_actions=24, state_dim=17, hidden_dim=64,
                 min_units=5, max_units=256, min_depth=2, max_depth=10):
        self.policy_net = torch.nn.Sequential(
            torch.nn.Linear(state_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, n_actions),
            torch.nn.Softmax(dim=-1)
        ).to(device)

        self.optimizer = torch.optim.Adam(self.policy_net.parameters(), lr=1e-3)
        self.gamma = 0.99

        self.min_units = min_units
        self.max_units = max_units
        self.min_depth = min_depth
        self.max_depth = max_depth

        self.min_base_points = 1000
        self.max_base_points = 10000
        self.max_refined_points = 10000

        self.episode_rewards = []
        self.episode_log_probs = []
        self.action_stats = {"weight": 0, "arch": 0, "sampling": 0, "total": 0}
        self.arch_action_stats = {
            "increase_depth": 0,
            "increase_width": 0,
            "decrease_depth": 0,
            "decrease_width": 0,
            "total_arch": 0,
            "total_decisions": 0,
        }

    def get_enhanced_state(self, losses, depth, units, loss_history,
                           darcy_base_points, ns_base_points,
                           darcy_refined_points, ns_refined_points):
        state = []
        state.extend(losses)
        state.append(depth / 10.0)
        state.append(units / 256.0)

        training_stability = 0.0
        if len(loss_history) > 5:
            recent_losses = loss_history[-5:]
            loss_std = np.std(recent_losses)
            if loss_std < 0.001:
                training_stability = 1.0
            elif loss_std > 0.1:
                training_stability = -1.0
        state.append(training_stability)

        improvement_trend = 0.0
        if len(loss_history) > 10:
            recent_10 = loss_history[-10:]
            improvement_trend = (recent_10[0] - recent_10[-1]) / (recent_10[0] + 1e-9)
        state.append(improvement_trend)

        medium_term_improvement = 0.0
        if len(loss_history) >= 50:
            recent_50 = loss_history[-50:]
            medium_term_improvement = (recent_50[0] - recent_50[-1]) / (recent_50[0] + 1e-9)
        state.append(medium_term_improvement)

        stagnation_indicator = 0.0
        if len(loss_history) >= 50:
            recent_50 = loss_history[-50:]
            improvement_50 = (recent_50[0] - recent_50[-1]) / (recent_50[0] + 1e-9)
            if abs(improvement_50) < 0.02:
                stagnation_indicator = 1.0
        state.append(stagnation_indicator)

        state.append(darcy_base_points / self.max_base_points)
        state.append(ns_base_points / self.max_base_points)
        state.append(darcy_refined_points / self.max_refined_points)
        state.append(ns_refined_points / self.max_refined_points)
        return torch.FloatTensor(state).to(device).unsqueeze(0)

    def get_valid_actions(self, depth, units, darcy_base_points, ns_base_points,
                          darcy_refined_points, ns_refined_points):
        valid = [1.0] * 24

        if depth >= self.max_depth:
            valid[14] = 0.0
        if units >= self.max_units:
            valid[15] = 0.0
        if depth <= self.min_depth:
            valid[16] = 0.0
        if units <= self.min_units:
            valid[17] = 0.0

        if darcy_refined_points >= self.max_refined_points:
            valid[18] = 0.0
        if darcy_base_points >= self.max_base_points:
            valid[19] = 0.0
        if darcy_base_points <= self.min_base_points:
            valid[20] = 0.0

        if ns_refined_points >= self.max_refined_points:
            valid[21] = 0.0
        if ns_base_points >= self.max_base_points:
            valid[22] = 0.0
        if ns_base_points <= self.min_base_points:
            valid[23] = 0.0

        return valid

    def select_action(self, state):
        state_tensor = state if isinstance(state, torch.Tensor) else torch.FloatTensor(state).to(device)
        probs = self.policy_net(state_tensor)

        state_values = state_tensor.squeeze(0)
        depth = int(state_values[7].item() * 10)
        units = int(state_values[8].item() * 256)
        darcy_base = int(state_values[11].item() * self.max_base_points)
        ns_base = int(state_values[12].item() * self.max_base_points)
        darcy_refined = int(state_values[13].item() * self.max_refined_points)
        ns_refined = int(state_values[14].item() * self.max_refined_points)

        valid_actions = self.get_valid_actions(
            depth, units, darcy_base, ns_base, darcy_refined, ns_refined
        )
        masked_probs = probs * torch.FloatTensor(valid_actions).to(device)
        masked_probs = masked_probs / (masked_probs.sum() + 1e-9)

        self.action_stats["total"] += 1
        action_dist = torch.distributions.Categorical(masked_probs)
        action = action_dist.sample()
        action_id = int(action.item())

        if action_id < 14:
            self.action_stats["weight"] += 1
        elif action_id < 18:
            self.action_stats["arch"] += 1
            self.arch_action_stats["total_decisions"] += 1
            self.arch_action_stats["total_arch"] += 1
            action_names = ["increase_depth", "increase_width", "decrease_depth", "decrease_width"]
            self.arch_action_stats[action_names[action_id - 14]] += 1
        else:
            self.action_stats["sampling"] += 1

        return action_id, action_dist.log_prob(action)

    def create_architecture_command(self, action_id):
        commands = {
            14: {"type": "depth", "direction": "increase"},
            15: {"type": "width", "direction": "increase"},
            16: {"type": "depth", "direction": "decrease"},
            17: {"type": "width", "direction": "decrease"},
        }
        return commands.get(action_id)

    def create_sampling_command(self, action_id):
        commands = {
            18: {"region": "darcy", "action": "add_refined"},
            19: {"region": "darcy", "action": "increase_base"},
            20: {"region": "darcy", "action": "decrease_base"},
            21: {"region": "ns", "action": "add_refined"},
            22: {"region": "ns", "action": "increase_base"},
            23: {"region": "ns", "action": "decrease_base"},
        }
        return commands.get(action_id)

    def update_policy(self):
        if len(self.episode_rewards) == 0:
            return

        discounted_rewards = []
        R = 0.0
        for reward in reversed(self.episode_rewards):
            R = reward + self.gamma * R
            discounted_rewards.insert(0, R)

        discounted_rewards = torch.tensor(discounted_rewards, dtype=torch.float32, device=device)
        if len(discounted_rewards) > 1:
            discounted_rewards = (discounted_rewards - discounted_rewards.mean()) / (discounted_rewards.std() + 1e-9)

        policy_loss = []
        for log_prob, ret in zip(self.episode_log_probs, discounted_rewards):
            policy_loss.append(-log_prob * ret)

        self.optimizer.zero_grad()
        torch.stack(policy_loss).sum().backward()
        self.optimizer.step()

        self.episode_rewards = []
        self.episode_log_probs = []

    def add_reward(self, reward):
        self.episode_rewards.append(reward)

    def add_log_prob(self, log_prob):
        self.episode_log_probs.append(log_prob)

    def print_statistics(self):
        if self.action_stats["total"] > 0:
            total = self.action_stats["total"]
            print(f"  weight adjustment: {self.action_stats['weight'] / total:.2%}")
            print(f"  architecture adjustment: {self.action_stats['arch'] / total:.2%}")
            print(f"  sampling adjustment: {self.action_stats['sampling'] / total:.2%}")


"""Physics-informed model and training logic"""
class PhysicsInformedNN:
    def __init__(self, X_darcy_t0, X_ns_t0, X_darcy_bc, X_ns_bc,
                 X_pressure_gauge, X_interface, X_darcy_lhs_pool, X_ns_lhs_pool,
                 use_rl=True):
        self.k11 = K[0][0]
        self.k22 = K[1][1]
        self.nu = nu
        self.g = g
        self.alpha = alpha
        self.s = s
        self.trace = trace

        self.X_darcy_initial = X_darcy_lhs_pool.copy()
        self.X_ns_initial = X_ns_lhs_pool.copy()
        self.N_f_darcy_current = len(self.X_darcy_initial)
        self.N_f_ns_current = len(self.X_ns_initial)
        self.N_darcy_refined = 0
        self.N_ns_refined = 0
        self.X_darcy_refined_current = np.empty((0, 3))
        self.X_ns_refined_current = np.empty((0, 3))

        self.X_darcy_t0_fixed = X_darcy_t0
        self.X_ns_t0_fixed = X_ns_t0
        self.X_darcy_bc_fixed = X_darcy_bc
        self.X_ns_bc_fixed = X_ns_bc
        self.X_pressure_gauge_fixed = X_pressure_gauge
        self.X_interface_fixed = X_interface

        self.phi_darcy_bc = torch.zeros((len(X_darcy_bc), 1), dtype=torch.float32, device=device)
        self.phi_t0 = torch.zeros((len(X_darcy_t0), 1), dtype=torch.float32, device=device)
        self.u1_t0 = torch.zeros((len(X_ns_t0), 1), dtype=torch.float32, device=device)
        self.u2_t0 = torch.zeros((len(X_ns_t0), 1), dtype=torch.float32, device=device)
        self.p_t0 = torch.zeros((len(X_ns_t0), 1), dtype=torch.float32, device=device)
        self.p_gauge = torch.zeros((len(X_pressure_gauge), 1), dtype=torch.float32, device=device)

        self.arch_config = {"depth": 4, "width": 64, "min_depth": 2, "max_depth": 10, "min_units": 5, "max_units": 256}
        self.build_network()
        self.optimizer = torch.optim.Adam(self.dnn.parameters(), lr=1e-3, weight_decay=1e-5)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode="min", factor=0.5, patience=500)

        self.weights = {
            "darcy": 10.0, "ns": 10.0, "interface": 10.0,
            "darcy_bc": 10.0, "ns_bc": 10.0,
            "darcy_t0": 10.0, "ns_t0": 10.0,
        }
        self._clip_weights()

        self.use_rl = use_rl
        if self.use_rl:
            self.rl_controller = RLController(
                n_actions=24, state_dim=17,
                min_units=self.arch_config["min_units"], max_units=self.arch_config["max_units"],
                min_depth=self.arch_config["min_depth"], max_depth=self.arch_config["max_depth"]
            )
            self.weight_adjust_interval = 200

        self.iter = 0
        self.prev_losses = None
        self.initial_losses = None
        self.adjustment_info = None
        self.weight_adjustment_info = None
        self.sampling_adjustment_info = None
        self.architecture_history = []
        self.sampling_history = []
        self.loss_history = []
        self.total_loss_history = []
        self.iteration_history = []
        self.current_losses = None
        self.cost_stats = {
            "rl_policy_inference_time": 0.0,
            "rl_policy_inference_calls": 0,
            "rl_policy_update_time": 0.0,
            "rl_policy_update_calls": 0,
        }
        self.computational_cost = {}

        self.update_residual_points()
        self.record_architecture(0, "Initial")

    def build_network(self):
        depth = self.arch_config["depth"]
        width = self.arch_config["width"]
        layers = [nn.Linear(3, width), nn.SiLU()]
        for _ in range(depth):
            layers += [nn.Linear(width, width), nn.SiLU()]
        layers += [nn.Linear(width, 4)]
        self.dnn = nn.Sequential(*layers).to(device)
        self.dnn.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def _clip_weights(self):
        for k in self.weights:
            self.weights[k] = max(min(float(self.weights[k]), 100.0), 1.0)

    def get_architecture_info(self):
        return {
            "depth": self.arch_config["depth"],
            "width": self.arch_config["width"],
            "total_params": count_trainable_parameters(self.dnn),
        }

    def record_architecture(self, step, action):
        info = self.get_architecture_info()
        info.update({"step": int(step), "action": action})
        self.architecture_history.append(info)

    def reset_optimizer(self):
        lr = self.optimizer.param_groups[0]["lr"]
        self.optimizer = torch.optim.Adam(self.dnn.parameters(), lr=lr, weight_decay=1e-5)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode="min", factor=0.5, patience=500)

    def migrate_weights(self, old_network, new_network):
        old_layers = [m for m in old_network.modules() if isinstance(m, nn.Linear)]
        new_layers = [m for m in new_network.modules() if isinstance(m, nn.Linear)]
        for old_layer, new_layer in zip(old_layers, new_layers):
            min_in = min(new_layer.in_features, old_layer.in_features)
            min_out = min(new_layer.out_features, old_layer.out_features)
            with torch.no_grad():
                new_layer.weight[:min_out, :min_in] = old_layer.weight[:min_out, :min_in]
                if new_layer.bias is not None and old_layer.bias is not None:
                    new_layer.bias[:min_out] = old_layer.bias[:min_out]

    def adjust_architecture(self, command):
        if command is None:
            return False
        success = False
        if command["type"] == "depth":
            if command["direction"] == "increase" and self.arch_config["depth"] < self.arch_config["max_depth"]:
                self.arch_config["depth"] += 1
                success = True
            elif command["direction"] == "decrease" and self.arch_config["depth"] > self.arch_config["min_depth"]:
                self.arch_config["depth"] -= 1
                success = True
        elif command["type"] == "width":
            old_w = self.arch_config["width"]
            new_w = min(int(old_w * 1.1), self.arch_config["max_units"]) if command["direction"] == "increase" else max(int(old_w * 0.9), self.arch_config["min_units"])
            if new_w != old_w:
                self.arch_config["width"] = new_w
                success = True
        if success:
            old_network = self.dnn
            self.build_network()
            self.migrate_weights(old_network, self.dnn)
            self.reset_optimizer()
            desc = f"{command['direction']} {command['type']}"
            self.record_architecture(self.iter, desc)
            print(f"Architecture adjusted: {desc}; depth={self.arch_config['depth']}, width={self.arch_config['width']}")
        return success

    def update_residual_points(self):
        self.darcy_indices = np.random.choice(len(self.X_darcy_initial), self.N_f_darcy_current, replace=False)
        self.ns_indices = np.random.choice(len(self.X_ns_initial), self.N_f_ns_current, replace=False)
        self.X_darcy_current_np = np.vstack([self.X_darcy_initial[self.darcy_indices], self.X_darcy_refined_current])
        self.X_ns_current_np = np.vstack([self.X_ns_initial[self.ns_indices], self.X_ns_refined_current])

        self.x_darcy, self.y_darcy, self.t_darcy = to_tensor_xyzt(self.X_darcy_current_np, True)
        self.x_ns, self.y_ns, self.t_ns = to_tensor_xyzt(self.X_ns_current_np, True)
        self.x_darcy_bc, self.y_darcy_bc, self.t_darcy_bc = to_tensor_xyzt(self.X_darcy_bc_fixed, True)
        self.x_ns_bc, self.y_ns_bc, self.t_ns_bc = to_tensor_xyzt(self.X_ns_bc_fixed, True)
        self.x_pressure_gauge, self.y_pressure_gauge, self.t_pressure_gauge = to_tensor_xyzt(self.X_pressure_gauge_fixed, True)
        self.x_interface, self.y_interface, self.t_interface = to_tensor_xyzt(self.X_interface_fixed, True)
        self.x_darcy_t0, self.y_darcy_t0, self.t_darcy_t0 = to_tensor_xyzt(self.X_darcy_t0_fixed, True)
        self.x_ns_t0, self.y_ns_t0, self.t_ns_t0 = to_tensor_xyzt(self.X_ns_t0_fixed, True)

    def net_forward(self, x, y, t):
        out = self.dnn(torch.cat([x, y, t], dim=1))
        return out[:, 0:1], out[:, 1:2], out[:, 2:3], out[:, 3:4]

    def physics_residuals(self, return_point_losses=False):

        phi, _, _, _ = self.net_forward(self.x_darcy, self.y_darcy, self.t_darcy)
        dphi_t = torch.autograd.grad(phi, self.t_darcy, torch.ones_like(phi), create_graph=True)[0]
        dphi_x = torch.autograd.grad(phi, self.x_darcy, torch.ones_like(phi), create_graph=True)[0]
        dphi_y = torch.autograd.grad(phi, self.y_darcy, torch.ones_like(phi), create_graph=True)[0]
        d2phi_x = torch.autograd.grad(dphi_x, self.x_darcy, torch.ones_like(dphi_x), create_graph=True)[0]
        d2phi_y = torch.autograd.grad(dphi_y, self.y_darcy, torch.ones_like(dphi_y), create_graph=True)[0]
        darcy_res = self.s * dphi_t - (self.k11 * d2phi_x + self.k22 * d2phi_y)
        darcy_point_losses = darcy_res**2
        loss_darcy = torch.mean(darcy_point_losses)

        _, u1, u2, p = self.net_forward(self.x_ns, self.y_ns, self.t_ns)
        du1_t = torch.autograd.grad(u1, self.t_ns, torch.ones_like(u1), create_graph=True)[0]
        du2_t = torch.autograd.grad(u2, self.t_ns, torch.ones_like(u2), create_graph=True)[0]
        du1_x = torch.autograd.grad(u1, self.x_ns, torch.ones_like(u1), create_graph=True)[0]
        du1_y = torch.autograd.grad(u1, self.y_ns, torch.ones_like(u1), create_graph=True)[0]
        du2_x = torch.autograd.grad(u2, self.x_ns, torch.ones_like(u2), create_graph=True)[0]
        du2_y = torch.autograd.grad(u2, self.y_ns, torch.ones_like(u2), create_graph=True)[0]
        d2u1_x = torch.autograd.grad(du1_x, self.x_ns, torch.ones_like(du1_x), create_graph=True)[0]
        d2u1_y = torch.autograd.grad(du1_y, self.y_ns, torch.ones_like(du1_y), create_graph=True)[0]
        d2u2_x = torch.autograd.grad(du2_x, self.x_ns, torch.ones_like(du2_x), create_graph=True)[0]
        d2u2_y = torch.autograd.grad(du2_y, self.y_ns, torch.ones_like(du2_y), create_graph=True)[0]
        dp_x = torch.autograd.grad(p, self.x_ns, torch.ones_like(p), create_graph=True)[0]
        dp_y = torch.autograd.grad(p, self.y_ns, torch.ones_like(p), create_graph=True)[0]
        continuity = du1_x + du2_y
        mom_x = du1_t + u1 * du1_x + u2 * du1_y - self.nu * (d2u1_x + d2u1_y) + dp_x
        mom_y = du2_t + u1 * du2_x + u2 * du2_y - self.nu * (d2u2_x + d2u2_y) + dp_y
        ns_point_losses = continuity**2 + mom_x**2 + mom_y**2
        loss_ns = torch.mean(ns_point_losses)

        phi_i, u1_i, u2_i, p_i = self.net_forward(self.x_interface, self.y_interface, self.t_interface)
        dphi_i_x = torch.autograd.grad(phi_i, self.x_interface, torch.ones_like(phi_i), create_graph=True)[0]
        dphi_i_y = torch.autograd.grad(phi_i, self.y_interface, torch.ones_like(phi_i), create_graph=True)[0]
        du1_i_y = torch.autograd.grad(u1_i, self.y_interface, torch.ones_like(u1_i), create_graph=True)[0]
        du2_i_x = torch.autograd.grad(u2_i, self.x_interface, torch.ones_like(u2_i), create_graph=True)[0]
        du2_i_y = torch.autograd.grad(u2_i, self.y_interface, torch.ones_like(u2_i), create_graph=True)[0]
        mass_res = -u2_i - self.k22 * dphi_i_y
        normal_res = p_i - 2.0 * self.nu * du2_i_y - self.g * phi_i
        bj_res = self.nu * (du1_i_y + du2_i_x) - self.alpha * self.nu / math.sqrt(self.trace) * (u1_i + self.k11 * dphi_i_x)
        loss_interface = torch.mean(mass_res**2 + normal_res**2 + bj_res**2)

        phi_bc, _, _, _ = self.net_forward(self.x_darcy_bc, self.y_darcy_bc, self.t_darcy_bc)
        loss_darcy_bc = torch.mean((phi_bc - self.phi_darcy_bc)**2)

        _, u1_bc, u2_bc, _ = self.net_forward(self.x_ns_bc, self.y_ns_bc, self.t_ns_bc)
        top_mask = (torch.abs(self.y_ns_bc - Y_NS_MAX) < 1e-6).float()
        u1_target = top_mask * (1.0 - torch.exp(-5.0 * self.t_ns_bc))
        u2_target = torch.zeros_like(u2_bc)
        _, _, _, p_g = self.net_forward(self.x_pressure_gauge, self.y_pressure_gauge, self.t_pressure_gauge)
        loss_ns_bc = torch.mean((u1_bc - u1_target)**2 + (u2_bc - u2_target)**2) + torch.mean((p_g - self.p_gauge)**2)

        phi_t0, _, _, _ = self.net_forward(self.x_darcy_t0, self.y_darcy_t0, self.t_darcy_t0)
        _, u1_t0, u2_t0, p_t0 = self.net_forward(self.x_ns_t0, self.y_ns_t0, self.t_ns_t0)
        loss_darcy_t0 = torch.mean((phi_t0 - self.phi_t0)**2)
        loss_ns_t0 = torch.mean((u1_t0 - self.u1_t0)**2 + (u2_t0 - self.u2_t0)**2 + 0.01 * (p_t0 - self.p_t0)**2)

        losses = (loss_darcy, loss_ns, loss_interface, loss_darcy_bc, loss_ns_bc, loss_darcy_t0, loss_ns_t0)
        if return_point_losses:
            return losses + (darcy_point_losses.detach(), ns_point_losses.detach())
        return losses

    def update_loss_history(self, total_loss):
        self.loss_history.append(total_loss)
        if len(self.loss_history) > 10000:
            self.loss_history.pop(0)

    def _adjust_base_points(self, region, increase):
        if region == "darcy":
            current_n = self.N_f_darcy_current
            min_n = self.rl_controller.min_base_points
            max_n = len(self.X_darcy_initial)
            new_n = min(int(current_n * 1.2), max_n) if increase else max(int(current_n * 0.8), min_n)
            if new_n != current_n:
                self.N_f_darcy_current = new_n
                return True
        else:
            current_n = self.N_f_ns_current
            min_n = self.rl_controller.min_base_points
            max_n = len(self.X_ns_initial)
            new_n = min(int(current_n * 1.2), max_n) if increase else max(int(current_n * 0.8), min_n)
            if new_n != current_n:
                self.N_f_ns_current = new_n
                return True
        return False

    def _add_refined_points(self, region):
        results = self.physics_residuals(return_point_losses=True)
        darcy_losses = results[-2].cpu().numpy().reshape(-1)
        ns_losses = results[-1].cpu().numpy().reshape(-1)
        if region == "darcy":
            if self.N_darcy_refined >= self.rl_controller.max_refined_points:
                return False
            Xcur, losses = self.X_darcy_current_np, darcy_losses[:len(self.X_darcy_current_np)]
            y0_domain, y1_domain = Y_D_MIN, Y_D_MAX
            n_new = min(1000, self.rl_controller.max_refined_points - self.N_darcy_refined)
        else:
            if self.N_ns_refined >= self.rl_controller.max_refined_points:
                return False
            Xcur, losses = self.X_ns_current_np, ns_losses[:len(self.X_ns_current_np)]
            y0_domain, y1_domain = Y_NS_MIN, Y_NS_MAX
            n_new = min(1000, self.rl_controller.max_refined_points - self.N_ns_refined)

        q = max(10, int(0.2 * len(losses)))
        box = Xcur[np.argsort(losses)[-q:]]
        lo = box.min(axis=0)
        hi = box.max(axis=0)
        pad = 0.05 * np.maximum(hi - lo, 1e-3)
        lo, hi = lo - pad, hi + pad
        lo[0], hi[0] = max(X_MIN, lo[0]), min(X_MAX, hi[0])
        lo[1], hi[1] = max(y0_domain, lo[1]), min(y1_domain, hi[1])
        lo[2], hi[2] = max(T_MIN, lo[2]), min(T_MAX, hi[2])
        new_points = lhs_box(n_new, lo[0], hi[0], lo[1], hi[1], lo[2], hi[2])
        if region == "darcy":
            self.X_darcy_refined_current = np.vstack([self.X_darcy_refined_current, new_points])
            self.N_darcy_refined = len(self.X_darcy_refined_current)
        else:
            self.X_ns_refined_current = np.vstack([self.X_ns_refined_current, new_points])
            self.N_ns_refined = len(self.X_ns_refined_current)
        return True

    def adjust_sampling_points(self, region, action):
        if action == "add_refined":
            return self._add_refined_points(region)
        if action == "increase_base":
            return self._adjust_base_points(region, True)
        if action == "decrease_base":
            return self._adjust_base_points(region, False)
        return False

    def loss_func(self):
        losses = self.physics_residuals(return_point_losses=False)
        self.current_losses = losses
        current_losses = [l.item() for l in losses]

        if self.initial_losses is None:
            self.initial_losses = current_losses[:]
        if self.prev_losses is None:
            self.prev_losses = current_losses[:]

        if self.use_rl and (self.iter % self.weight_adjust_interval == 0):
            arch_info = self.get_architecture_info()
            t0 = time.perf_counter()
            state = self.rl_controller.get_enhanced_state(
                current_losses,
                arch_info["depth"],
                arch_info["width"],
                self.loss_history,
                self.N_f_darcy_current,
                self.N_f_ns_current,
                self.N_darcy_refined,
                self.N_ns_refined,
            )
            action_id, log_prob = self.rl_controller.select_action(state)
            self.cost_stats["rl_policy_inference_time"] += time.perf_counter() - t0
            self.cost_stats["rl_policy_inference_calls"] += 1

            if action_id < 14:
                action_map = {0: 10.0, 1: -10.0}
                adjust_factor = action_map[action_id % 2]
                target_loss_idx = action_id // 2
                loss_keys = ["darcy", "ns", "interface", "darcy_bc", "ns_bc", "darcy_t0", "ns_t0"]
                selected_key = loss_keys[target_loss_idx]
                self.weights[selected_key] += adjust_factor
                self._clip_weights()
                self.weight_adjustment_info = {
                    "step": self.iter,
                    "pre_losses": current_losses[:],
                    "log_prob": log_prob,
                    "selected_key": selected_key,
                }
                self.rl_controller.add_log_prob(log_prob)
            elif action_id < 18:
                command = self.rl_controller.create_architecture_command(action_id)
                self.adjustment_info = {
                    "step": self.iter,
                    "pre_losses": current_losses[:],
                    "arch_action": action_id,
                    "log_prob": log_prob,
                    "command": command,
                }
                self.rl_controller.add_log_prob(log_prob)
                if self.adjust_architecture(command):
                    action_desc = f"{command['direction']} {command['type']}"
                    self.adjustment_info["action_desc"] = action_desc
                else:
                    self.rl_controller.add_reward(-0.1)
                    self.adjustment_info = None
            else:
                command = self.rl_controller.create_sampling_command(action_id)
                if command:
                    self.sampling_adjustment_info = {
                        "step": self.iter,
                        "pre_losses": current_losses[:],
                        "log_prob": log_prob,
                        "command": command,
                    }
                    self.rl_controller.add_log_prob(log_prob)
                    success = self.adjust_sampling_points(command["region"], command["action"])
                    if success:
                        self.update_residual_points()
                        self.sampling_history.append({
                            "step": self.iter,
                            "command": command,
                            "darcy_base": self.N_f_darcy_current,
                            "ns_base": self.N_f_ns_current,
                            "darcy_refined": self.N_darcy_refined,
                            "ns_refined": self.N_ns_refined,
                        })
                    else:
                        self.rl_controller.add_reward(-0.1)
                        self.sampling_adjustment_info = None

        if self.use_rl and self.weight_adjustment_info and (self.iter - self.weight_adjustment_info["step"]) >= 200:
            weight_reward = calculate_reward(
                current_losses,
                self.weight_adjustment_info["pre_losses"],
                self.initial_losses,
            )
            self.rl_controller.add_reward(weight_reward)
            self.weight_adjustment_info = None

        if self.use_rl and self.adjustment_info and (self.iter - self.adjustment_info["step"]) >= 200:
            arch_info = self.get_architecture_info()
            arch_reward = calculate_arch_reward(
                self.adjustment_info["pre_losses"],
                current_losses,
                arch_info["total_params"],
                self.adjustment_info.get("action_desc"),
                arch_info["depth"],
                arch_info["width"],
            )
            self.rl_controller.add_reward(arch_reward)
            self.adjustment_info = None

        if self.use_rl and self.sampling_adjustment_info:
            steps_elapsed = self.iter - self.sampling_adjustment_info["step"]
            if steps_elapsed == 200:
                sampling_reward = calculate_reward(
                    current_losses,
                    self.sampling_adjustment_info["pre_losses"],
                    self.initial_losses,
                )
                self.rl_controller.add_reward(sampling_reward)
                self.sampling_adjustment_info = None

        total_loss = sum([
            self.weights["darcy"] * losses[0],
            self.weights["ns"] * losses[1],
            self.weights["interface"] * losses[2],
            self.weights["darcy_bc"] * losses[3],
            self.weights["ns_bc"] * losses[4],
            self.weights["darcy_t0"] * losses[5],
            self.weights["ns_t0"] * losses[6],
        ])

        self.update_loss_history(total_loss.item())
        if self.iter % 10 == 0:
            self.total_loss_history.append(total_loss.item())
            self.iteration_history.append(self.iter)
        if self.iter % 100 == 0:
            self.scheduler.step(total_loss.item())

        if self.use_rl and (self.iter % 500 == 0) and len(self.rl_controller.episode_rewards) > 0:
            self.rl_controller.episode_rewards = [r * 0.1 for r in self.rl_controller.episode_rewards]
            t0 = time.perf_counter()
            self.rl_controller.update_policy()
            self.cost_stats["rl_policy_update_time"] += time.perf_counter() - t0
            self.cost_stats["rl_policy_update_calls"] += 1

        self.prev_losses = current_losses[:]
        self.iter += 1
        return total_loss


    def train(self, stage1_epochs=STAGE1_EPOCHS, stage2_epochs=STAGE2_EPOCHS):
        print("=" * 80)
        print("Single-network Auto-PINNs: transient lid-driven cavity over porous bed")
        print("=" * 80)

        sync_cuda_if_available()
        training_start = time.time()
        self.dnn.train()

        print(f"\nStage 1: RL dynamic adaptation ({stage1_epochs} steps)")
        reset_cuda_peak_memory()
        stage1_start = time.time()
        stage1_losses = []
        self.use_rl = True
        for epoch in range(stage1_epochs):
            self.optimizer.zero_grad()
            loss = self.loss_func()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.dnn.parameters(), max_norm=1.0)
            self.optimizer.step()
            stage1_losses.append(loss.item())
            if epoch % 1000 == 0:
                arch = self.get_architecture_info()
                print(
                    f"\nStage1 {epoch}/{stage1_epochs} | loss={loss.item():.3e} | "
                    f"D={arch['depth']}, W={arch['width']} | "
                    f"Darcy={self.N_f_darcy_current}+{self.N_darcy_refined}, "
                    f"NS={self.N_f_ns_current}+{self.N_ns_refined} | "
                    f"weights={self.weights}"
                )
        sync_cuda_if_available()
        stage1_duration = time.time() - stage1_start
        stage1_memory = get_cuda_memory_summary()

        stage1_final_config = {
            "depth": self.arch_config["depth"],
            "width": self.arch_config["width"],
            "weights": self.weights.copy(),
            "N_darcy_base": self.N_f_darcy_current,
            "N_ns_base": self.N_f_ns_current,
            "N_darcy_refined": self.N_darcy_refined,
            "N_ns_refined": self.N_ns_refined,
            "final_loss": stage1_losses[-1],
            "training_time": stage1_duration,
        }

        print(f"\nStage 2: fixed-configuration fine tuning ({stage2_epochs} steps)")
        reset_cuda_peak_memory()
        stage2_start = time.time()
        stage2_losses = []
        self.use_rl = False
        for epoch in range(stage2_epochs):
            self.optimizer.zero_grad()
            loss = self.loss_func()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.dnn.parameters(), max_norm=1.0)
            self.optimizer.step()
            stage2_losses.append(loss.item())
            if epoch % 1000 == 0:
                arch = self.get_architecture_info()
                print(
                    f"\nStage2 {epoch}/{stage2_epochs} | loss={loss.item():.3e} | "
                    f"D={arch['depth']}, W={arch['width']} | "
                    f"Darcy={self.N_f_darcy_current}+{self.N_darcy_refined}, "
                    f"NS={self.N_f_ns_current}+{self.N_ns_refined}"
                )
        sync_cuda_if_available()
        stage2_duration = time.time() - stage2_start
        stage2_memory = get_cuda_memory_summary()
        total_training_time = time.time() - training_start

        self.computational_cost = self.build_computational_cost_report(
            stage1_duration, stage2_duration, total_training_time,
            stage1_memory, stage2_memory, stage1_epochs, stage2_epochs, stage1_final_config
        )
        write_computational_cost_report(self.computational_cost, COST_REPORT_PREFIX)

        print("\nTraining completed")
        print(f"Stage1: {stage1_duration:.1f}s, Stage2: {stage2_duration:.1f}s, Total: {total_training_time:.1f}s")
        print(f"Final config: {stage1_final_config}")

        return {
            "stage1_losses": stage1_losses,
            "stage2_losses": stage2_losses,
            "stage1_config": stage1_final_config,
            "stage1_time": stage1_duration,
            "stage2_time": stage2_duration,
            "total_time": total_training_time,
            "computational_cost": self.computational_cost,
        }

    def build_computational_cost_report(self, stage1_duration, stage2_duration, total_training_time,
                                        stage1_memory, stage2_memory, stage1_epochs, stage2_epochs, stage1_final_config):
        final_training_points = int(
            self.N_f_darcy_current + self.N_darcy_refined +
            self.N_f_ns_current + self.N_ns_refined +
            len(self.X_darcy_bc_fixed) + len(self.X_ns_bc_fixed) +
            len(self.X_pressure_gauge_fixed) + len(self.X_interface_fixed) +
            len(self.X_darcy_t0_fixed) + len(self.X_ns_t0_fixed)
        )
        peak_allocated = None
        peak_reserved = None
        if stage1_memory.get("cuda_available") or stage2_memory.get("cuda_available"):
            peak_allocated = max(stage1_memory.get("peak_allocated_mb") or 0, stage2_memory.get("peak_allocated_mb") or 0)
            peak_reserved = max(stage1_memory.get("peak_reserved_mb") or 0, stage2_memory.get("peak_reserved_mb") or 0)
        rl_inf = self.cost_stats["rl_policy_inference_time"]
        rl_upd = self.cost_stats["rl_policy_update_time"]
        return {
            "problem": "Transient lid-driven cavity over a porous bed (Navier-NS-Darcy with BJ interface)",
            "method": "Single-network Auto-PINNs with RL-guided loss weights, architecture and adaptive sampling",
            "hardware": get_hardware_info(),
            "model": {
                "architecture_depth": int(self.arch_config["depth"]),
                "architecture_width": int(self.arch_config["width"]),
                "trainable_parameters": count_trainable_parameters(self.dnn),
            },
            "training": {
                "stage1_epochs": int(stage1_epochs),
                "stage2_epochs": int(stage2_epochs),
                "stage1_time_s": float(stage1_duration),
                "stage2_time_s": float(stage2_duration),
                "total_training_time_s": float(total_training_time),
                "wall_clock_hours": float(total_training_time / 3600.0),
                "gpu_hours": float(total_training_time / 3600.0) if torch.cuda.is_available() else None,
                "cpu_hours": None if torch.cuda.is_available() else float(total_training_time / 3600.0),
                "final_training_points": final_training_points,
            },
            "sampling": {
                "darcy_base_points": int(self.N_f_darcy_current),
                "darcy_refined_points": int(self.N_darcy_refined),
                "ns_base_points": int(self.N_f_ns_current),
                "ns_refined_points": int(self.N_ns_refined),
                "darcy_bc_points": int(len(self.X_darcy_bc_fixed)),
                "ns_bc_points": int(len(self.X_ns_bc_fixed)),
                "pressure_gauge_points": int(len(self.X_pressure_gauge_fixed)),
                "interface_points": int(len(self.X_interface_fixed)),
                "darcy_t0_points": int(len(self.X_darcy_t0_fixed)),
                "ns_t0_points": int(len(self.X_ns_t0_fixed)),
            },
            "final_rl_configuration": {
                "stage1_final_config": stage1_final_config,
                "architecture_history": self.architecture_history,
                "sampling_history": self.sampling_history,
            },
            "memory": {
                "stage1_cuda_memory": stage1_memory,
                "stage2_cuda_memory": stage2_memory,
                "training_peak_allocated_mb": peak_allocated,
                "training_peak_reserved_mb": peak_reserved,
                "cpu_rss_mb_at_report": get_cpu_memory_mb(),
            },
            "rl_overhead": {
                "policy_inference_time_s": float(rl_inf),
                "policy_inference_calls": int(self.cost_stats["rl_policy_inference_calls"]),
                "policy_update_time_s": float(rl_upd),
                "policy_update_calls": int(self.cost_stats["rl_policy_update_calls"]),
                "total_measured_overhead_time_s": float(rl_inf + rl_upd),
                "overhead_fraction_of_training_time": float((rl_inf + rl_upd) / total_training_time) if total_training_time > 0 else None,
            },
            "inference": {},
        }

    def finalize_computational_cost_report(self, prediction_duration=None, prediction_points=None):
        inference = self.computational_cost.setdefault("inference", {})
        if prediction_duration is not None:
            inference["prediction_time_s"] = float(prediction_duration)
        if prediction_points is not None:
            inference["prediction_points"] = int(prediction_points)
            if prediction_duration is not None and prediction_points > 0:
                inference["prediction_time_per_point_s"] = float(prediction_duration / prediction_points)
        write_computational_cost_report(self.computational_cost, COST_REPORT_PREFIX)

    def predict(self, XYT, batch_size=20000):
        self.dnn.eval()
        outs = []
        with torch.no_grad():
            for i in range(0, len(XYT), batch_size):
                Xb = XYT[i:i+batch_size]
                x = torch.tensor(Xb[:, 0:1], dtype=torch.float32, device=device)
                y = torch.tensor(Xb[:, 1:2], dtype=torch.float32, device=device)
                t = torch.tensor(Xb[:, 2:3], dtype=torch.float32, device=device)
                phi, u1, u2, p = self.net_forward(x, y, t)
                outs.append(torch.cat([phi, u1, u2, p], dim=1).cpu().numpy())
        out = np.vstack(outs)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3], out[:, 3:4]

X_darcy_lhs_pool = lhs_box(N_BASE_DARCY, X_MIN, X_MAX, Y_D_MIN, Y_D_MAX)
X_ns_lhs_pool = lhs_box(N_BASE_STOKES, X_MIN, X_MAX, Y_NS_MIN, Y_NS_MAX)

n_top = N_BC_STOKES // 3
n_left_s = N_BC_STOKES // 3
n_right_s = N_BC_STOKES - n_top - n_left_s
X_ns_bc = np.vstack([sample_line(n_top, "top"), sample_line(n_left_s, "left_s"), sample_line(n_right_s, "right_s")])

n_bottom = N_BC_DARCY // 3
n_left_d = N_BC_DARCY // 3
n_right_d = N_BC_DARCY - n_bottom - n_left_d
X_darcy_bc = np.vstack([sample_line(n_bottom, "bottom"), sample_line(n_left_d, "left_d"), sample_line(n_right_d, "right_d")])

X_interface = sample_line(N_INTERFACE, "interface")
X_pressure_gauge = np.hstack([np.ones((200, 1)) * 0.5, np.ones((200, 1)) * 0.25, np.random.rand(200, 1)])

X_darcy_t0 = lhs_box(N_T0_DARCY, X_MIN, X_MAX, Y_D_MIN, Y_D_MAX, 0.0, 0.0)
X_ns_t0 = lhs_box(N_T0_STOKES, X_MIN, X_MAX, Y_NS_MIN, Y_NS_MAX, 0.0, 0.0)

model = PhysicsInformedNN(
    X_darcy_t0, X_ns_t0,
    X_darcy_bc, X_ns_bc, X_pressure_gauge, X_interface,
    X_darcy_lhs_pool, X_ns_lhs_pool,
    use_rl=True,
)

training_results = model.train(stage1_epochs=STAGE1_EPOCHS, stage2_epochs=STAGE2_EPOCHS)

print("\nFinal dynamic configuration:")
print(f"  Architecture: depth={model.arch_config['depth']}, width={model.arch_config['width']}")
print(f"  Loss weights: {model.weights}")
print(f"  Darcy sampling: base={model.N_f_darcy_current}, refined={model.N_darcy_refined}")
print(f"  NS sampling: base={model.N_f_ns_current}, refined={model.N_ns_refined}")


sync_cuda_if_available()
pred_start = time.time()
prediction_points = 0
velocity_summary_rows = []

"""Multi-time visualization"""
for t_target in [0.25, 0.50, 1.00]:
    nx, ny = 121, 121
    xs = np.linspace(X_MIN, X_MAX, nx)
    ys = np.linspace(Y_D_MIN, Y_NS_MAX, ny)
    Xg, Yg = np.meshgrid(xs, ys)
    Tg = np.ones_like(Xg) * t_target
    X_star = np.hstack([Xg.reshape(-1, 1), Yg.reshape(-1, 1), Tg.reshape(-1, 1)])
    prediction_points += len(X_star)
    phi_pred, u1_pred, u2_pred, p_pred = model.predict(X_star)
    phi = phi_pred.reshape(ny, nx)
    u1 = u1_pred.reshape(ny, nx)
    u2 = u2_pred.reshape(ny, nx)
    p = p_pred.reshape(ny, nx)

    dphi_dy, dphi_dx = np.gradient(phi, ys, xs)
    U = np.where(Yg >= 0.0, u1, -K[0][0] * dphi_dx)
    V = np.where(Yg >= 0.0, u2, -K[1][1] * dphi_dy)
    speed = np.sqrt(U**2 + V**2)
    ns_mask = Yg >= 0.0
    darcy_mask = Yg < 0.0
    velocity_summary = {
        "time": float(t_target),
        "global_speed_min": float(np.nanmin(speed)),
        "global_speed_max": float(np.nanmax(speed)),
        "global_speed_mean": float(np.nanmean(speed)),
        "ns_speed_max": float(np.nanmax(speed[ns_mask])),
        "ns_speed_mean": float(np.nanmean(speed[ns_mask])),
        "darcy_speed_max": float(np.nanmax(speed[darcy_mask])),
        "darcy_speed_mean": float(np.nanmean(speed[darcy_mask])),
    }
    velocity_summary_rows.append(velocity_summary)
    print(
        f"Velocity values t={t_target:.2f}: "
        f"global max={velocity_summary['global_speed_max']:.6e}, "
        f"global mean={velocity_summary['global_speed_mean']:.6e}, "
        f"Darcy max={velocity_summary['darcy_speed_max']:.6e}, "
        f"Darcy mean={velocity_summary['darcy_speed_mean']:.6e}"
    )
    np.savez_compressed(
        os.path.join(OUTPUT_DIR, f"global_velocity_values_t{t_target:.2f}_k1e-5.npz"),
        x=xs,
        y=ys,
        U=U,
        V=V,
        speed=speed,
        phi=phi,
        p=p,
    )
    velocity_table = np.column_stack([
        Xg.reshape(-1), Yg.reshape(-1), Tg.reshape(-1),
        U.reshape(-1), V.reshape(-1), speed.reshape(-1),
    ])
    np.savetxt(
        os.path.join(OUTPUT_DIR, f"global_velocity_values_t{t_target:.2f}_k1e-5.csv"),
        velocity_table,
        delimiter=",",
        header="x,y,t,U,V,speed",
        comments="",
    )

    fig, ax = plt.subplots(figsize=(8, 7))
    cf = ax.contourf(Xg, Yg, speed, levels=60, cmap="coolwarm")
    ax.streamplot(xs, ys, U, V, color="white", density=1.2, linewidth=0.7, arrowsize=0.8)
    ax.axhline(0.0, color="red", linestyle="--", linewidth=1.5)
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_D_MIN, Y_NS_MAX)
    ax.margins(0.0)
    ax.set_title(f"Lid-driven cavity over porous bed, t={t_target:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal", adjustable="box")
    fig.colorbar(cf, ax=ax, label="|velocity|")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"lid_cavity_porous_velocity_t{t_target:.2f}.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, field, title in zip(axes, [phi, p, speed], ["phi / hydraulic head", "pressure p", "velocity magnitude"]):
        im = ax.contourf(Xg, Yg, field, levels=50, cmap="coolwarm")
        ax.axhline(0.0, color="k", linestyle="--", linewidth=1.0)
        ax.set_xlim(X_MIN, X_MAX)
        ax.set_ylim(Y_D_MIN, Y_NS_MAX)
        ax.margins(0.0)
        ax.set_title(title)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_aspect("equal", adjustable="box")
        fig.colorbar(im, ax=ax)
    fig.suptitle(f"Predicted fields at t={t_target:.2f}")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"lid_cavity_porous_fields_t{t_target:.2f}.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)

if velocity_summary_rows:
    velocity_summary_path = os.path.join(OUTPUT_DIR, "global_velocity_summary_k1e-5.csv")
    with open(velocity_summary_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(velocity_summary_rows[0].keys()))
        writer.writeheader()
        writer.writerows(velocity_summary_rows)
    print(f"Velocity summary saved to: {velocity_summary_path}")

sync_cuda_if_available()
prediction_duration = time.time() - pred_start
model.finalize_computational_cost_report(prediction_duration=prediction_duration, prediction_points=prediction_points)

plt.figure(figsize=(10, 5))
plt.semilogy(model.iteration_history, model.total_loss_history, linewidth=1.8)
plt.xlabel("Iteration")
plt.ylabel("Total loss (log scale)")
plt.title("Training loss: lid-driven cavity over porous bed")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "lid_cavity_porous_training_loss.png"), dpi=300, bbox_inches="tight")
plt.close()

print(f"Prediction/visualization completed in {prediction_duration:.2f}s. Results saved to: {OUTPUT_DIR}")

